# 🚦 AI-Based Traffic Violation Detection with License Plate Recognition

**Hệ thống phát hiện vi phạm giao thông và nhận diện biển số xe sử dụng YOLOv8 + OCR**

### 🧠 Cách tiếp cận (cập nhật theo backend):
- **YOLOv8n**: Phát hiện phương tiện (xe máy, ô tô, xe tải, bus) + crop có ngữ cảnh
- **license_plate_detector.pt + FastPlateOCR**: Nhận diện biển số xe
- **vehicle_color_n_cls.pt**: Phân loại màu xe (15 màu)
- **helmet.pt**: Phát hiện vi phạm trên TOÀN BỘ ẢNH (full-image): không mũ, điện thoại, chở quá người
- **traffic_light.pt + HSV zone-based**: Phát hiện đèn giao thông + phân tích màu theo vùng
- **Zone-based Red Light Detection**: Phát hiện vượt đèn đỏ dựa trên vùng (Waiting → Stop → Intersection)
- **Bird's Eye View (BEV) 3D**: Perspective Transform sang không gian 3D, hình thang + vạch dừng ảo
- **Centroid Tracking**: Theo dõi xe qua các frame

---
## ⚙️ 1. Cài đặt môi trường & Kết nối Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/TrafficAI'
for sub in ['models','results','videos','images']:
    os.makedirs(f'{DRIVE_BASE}/{sub}', exist_ok=True)
print(f'✅ Drive mounted! Working: {DRIVE_BASE}')

In [ ]:
!pip install -q ultralytics opencv-python fast-plate-ocr numpy pillow matplotlib tqdm
print('✅ Dependencies installed!')

In [ ]:
MODEL_DIR = f'{DRIVE_BASE}/models'
def check_models():
    required = ['license_plate_detector.pt', 'yolov8n.pt', 'vehicle_color_n_cls.pt', 'helmet.pt', 'traffic_light.pt']
    missing = [m for m in required if not os.path.exists(f'{MODEL_DIR}/{m}')]
    if 'yolov8n.pt' in missing:
        from ultralytics import YOLO
        YOLO('yolov8n.pt')
        import shutil
        shutil.move('yolov8n.pt', f'{MODEL_DIR}/yolov8n.pt')
        missing.remove('yolov8n.pt')
        print('✅ yolov8n.pt downloaded')
    if not missing:
        print('✅ All models ready!')
    else:
        print(f'❌ Missing: {missing}')
        print(f'📤 Upload to: {MODEL_DIR}/')
    return len(missing) == 0
check_models()

---
## 🧠 2. Khởi tạo Models & Functions

In [ ]:
import cv2, numpy as np, time, re, os, base64
from datetime import datetime
from tqdm.notebook import tqdm
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt
from ultralytics import YOLO
from fast_plate_ocr import LicensePlateRecognizer as FastPlateOCR
from google.colab import files

print('✅ Libraries imported!')

In [ ]:
# === Load Models ===
yolo_plate = YOLO(f'{MODEL_DIR}/license_plate_detector.pt')
yolo_vehicle = YOLO(f'{MODEL_DIR}/yolov8n.pt')

try:
    color_model = YOLO(f'{MODEL_DIR}/vehicle_color_n_cls.pt')
    print('✅ Color model loaded')
except Exception as e:
    print(f'⚠️ Color model not loaded: {e}')
    color_model = None

try:
    helmet_model = YOLO(f'{MODEL_DIR}/helmet.pt')
    print('✅ Violation model (helmet.pt) loaded')
except Exception as e:
    print(f'⚠️ Violation model not loaded: {e}')
    helmet_model = None

try:
    light_model = YOLO(f'{MODEL_DIR}/traffic_light.pt')
    print('✅ Traffic light model loaded')
except Exception as e:
    print(f'⚠️ Traffic light model not loaded: {e}')
    light_model = None

ocr = FastPlateOCR(hub_ocr_model='cct-s-v2-global-model', device='auto')
print('✅ OCR engine loaded')
print('\n🟢 All models ready!')

In [ ]:
# ======================== CONSTANTS ========================
COLOR_NAMES = ['beige','black','blue','brown','gold','green','grey','orange','pink','purple','red','silver','tan','white','yellow']
VEHICLE_CLASSES = {2:'car', 3:'motorcycle', 5:'bus', 7:'truck'}
TRAFFIC_ROAD_USER_CLASSES = {'biker', 'car', 'pedestrian', 'truck'}

# Violation colors (BGR)
VIOLATION_COLORS = {
    'MORE_THAN_TWO_PERSONS': (0, 0, 255),    # Đỏ
    'WITHOUT_HELMET': (0, 165, 255),          # Cam
    'USING_MOBILE': (255, 0, 255),            # Tím (Magenta)
    'RED_LIGHT_VIOLATION': (0, 0, 255)        # Đỏ đậm
}

# Traffic light state colors (BGR)
STATE_COLORS = {
    'red': (0, 0, 255),
    'yellow': (0, 255, 255),
    'green': (0, 200, 0),
    'unknown': (180, 180, 180)
}

# Zone colors for visualization (BGR)
ZONE_COLORS = {
    'waiting': (200, 200, 100),      # Vàng nhạt
    'stop': (100, 100, 200),         # Đỏ nhạt
    'intersection': (100, 200, 100), # Xanh lá nhạt
}

print('✅ Constants defined!')

In [ ]:
# ======================== CORE DETECTION FUNCTIONS ========================

def crop_vehicle_context(image, bbox, vehicle_type):
    """
    Crop có ngữ cảnh. Với xe máy, mở rộng lên trên và hai bên
    để chứa người ngồi, mũ và tay.
    """
    x1, y1, x2, y2 = bbox
    h_img, w_img = image.shape[:2]
    width = max(1, x2 - x1)
    height = max(1, y2 - y1)

    if vehicle_type == 'motorcycle':
        pad_left = int(width * 0.45)
        pad_right = int(width * 0.45)
        pad_top = int(height * 1.20)
        pad_bottom = int(height * 0.25)
    else:
        pad_left = pad_right = pad_top = pad_bottom = 0

    cx1 = max(0, x1 - pad_left)
    cy1 = max(0, y1 - pad_top)
    cx2 = min(w_img, x2 + pad_right)
    cy2 = min(h_img, y2 + pad_bottom)
    crop = image[cy1:cy2, cx1:cx2]
    return crop, (cx1, cy1, cx2, cy2)


def detect_vehicles(image, conf=0.25):
    """
    Phát hiện phương tiện trong ảnh.
    Trả về list các tuple: (bbox, vtype, color, conf, vehicle_crop, crop_bbox)
    """
    results = yolo_vehicle.predict(
        image, device='cpu', classes=list(VEHICLE_CLASSES.keys()), conf=conf
    )[0]
    vehicles = []
    if results.boxes is None:
        return vehicles
    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls_id = int(box.cls[0])
        conf_val = float(box.conf[0])
        vtype = VEHICLE_CLASSES.get(cls_id, 'vehicle')

        vehicle_crop, crop_bbox = crop_vehicle_context(image, (x1, y1, x2, y2), vtype)

        # Phân loại màu
        color = 'unknown'
        if color_model is not None and vehicle_crop.size > 0:
            try:
                cr = color_model.predict(vehicle_crop, device='cpu', verbose=False)[0]
                if hasattr(cr, 'probs') and cr.probs is not None:
                    idx = cr.probs.top1
                    color = COLOR_NAMES[idx] if idx < len(COLOR_NAMES) else 'unknown'
                elif hasattr(cr, 'boxes') and cr.boxes is not None and len(cr.boxes) > 0:
                    cls_id_color = int(cr.boxes.cls[0])
                    color = COLOR_NAMES[cls_id_color] if cls_id_color < len(COLOR_NAMES) else 'unknown'
            except:
                pass

        vehicles.append(((x1, y1, x2, y2), vtype, color, conf_val, vehicle_crop, crop_bbox))
    return vehicles


def detect_plates(image, conf=0.1):
    """Phát hiện biển số trong ảnh. Trả về list (plate_img, bbox)."""
    results = yolo_plate.predict(image, device='cpu', conf=conf)[0]
    plates = []
    if results.boxes is None:
        return plates
    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        h, w = image.shape[:2]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        plate_img = image[y1:y2, x1:x2]
        plates.append((plate_img, (x1, y1, x2, y2)))
    return plates


def extract_text(plate_img):
    """Trích xuất văn bản từ ảnh biển số bằng OCR."""
    if plate_img is None or plate_img.size == 0:
        return ''
    image_rgb = cv2.cvtColor(plate_img, cv2.COLOR_BGR2RGB)
    result = ocr.run(image_rgb)
    raw_text = ''
    if hasattr(result, 'text'):
        raw_text = result.text
    elif isinstance(result, str):
        raw_text = result
    elif isinstance(result, list) and len(result) > 0:
        first = result[0]
        if hasattr(first, 'text'):
            raw_text = first.text
        else:
            raw_text = str(first)
    else:
        raw_text = str(result) if result else ''

    keywords = ['PREDICTION', 'PLATE', 'CHARPROBS', 'CHARS', 'REGION',
                'UNITEDKINGDOM', 'VIETNAM', 'NONE', 'PROB', 'DETECTION', 'CONFIDENCE']
    for kw in keywords:
        raw_text = re.sub(kw, '', raw_text, flags=re.IGNORECASE)

    cleaned = re.sub(r'[^A-Z0-9\-.]', '', raw_text.upper())
    # Pattern biển số Việt Nam: 2 số + 1-2 chữ + số
    m = re.search(r'(\d{1,2}[A-Z]{1,2}[-\d\.]*\d+)', cleaned)
    if m: return m.group(1)
    # Pattern châu Âu
    m = re.search(r'([A-Z]{1,3}[0-9]{1,4}[A-Z]{0,3})', cleaned)
    if m: return m.group(1)
    # Fallback
    m = re.search(r'([A-Z0-9]{4,})', cleaned)
    return m.group(1) if m else cleaned


def match_plates_to_vehicles(vehicles, plates):
    """Ghép biển số với xe dựa trên tọa độ (plate nằm trong vehicle bbox)."""
    matched = []
    for v in vehicles:
        bbox, vtype, color, conf, vehicle_crop, crop_bbox = v
        vx1, vy1, vx2, vy2 = bbox
        for plate_img, (px1, py1, px2, py2) in plates:
            if px1 >= vx1 and px2 <= vx2 and py1 >= vy1 and py2 <= vy2:
                plate_text = extract_text(plate_img)
                if plate_text:
                    matched.append((bbox, vtype, color, plate_img, plate_text))
                break
    return matched


def detect_violations_on_full_image(full_image, debug=False):
    """
    Phát hiện vi phạm trên TOÀN BỘ ẢNH (full-image) - MỘT LẦN DUY NHẤT.
    Sử dụng helmet.pt để phát hiện: không mũ, điện thoại, chở quá người.
    
    Trả về: list các dict với keys: type, details, bbox, conf, bottom_center
    """
    violations = []
    if helmet_model is None or full_image is None or full_image.size == 0:
        return violations

    results = helmet_model.predict(
        full_image, device='cpu', conf=0.15, iou=0.45,
        agnostic_nms=True, verbose=False
    )

    if len(results) == 0 or results[0].boxes is None:
        return violations

    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        class_id = int(box.cls[0])
        class_name = results[0].names[class_id]
        conf_val = float(box.conf[0])
        cn = class_name.strip().lower()

        is_valid = False
        viol_type = None
        details = None

        if 'more_than_two_persons' in cn or 'more_than' in cn:
            if conf_val >= 0.50:  # violation_conf_more_than_two
                is_valid = True
                viol_type = 'MORE_THAN_TWO_PERSONS'
                details = 'Xe chở quá 2 người'
        elif 'without_helmet' in cn or 'no_helmet' in cn or 'w/o_helmet' in cn:
            if conf_val >= 0.25:  # violation_conf_without_helmet
                is_valid = True
                viol_type = 'WITHOUT_HELMET'
                details = 'Người không đội mũ bảo hiểm'
        elif 'using_mobile' in cn or 'phone' in cn or 'mobile' in cn:
            if conf_val >= 0.25:  # violation_conf_using_mobile
                is_valid = True
                viol_type = 'USING_MOBILE'
                details = 'Sử dụng điện thoại khi lái xe'

        if is_valid:
            bottom_cx = (x1 + x2) / 2.0
            bottom_cy = float(y2)
            violations.append({
                'type': viol_type,
                'details': details,
                'bbox': (x1, y1, x2, y2),
                'conf': conf_val,
                'bottom_center': (bottom_cx, bottom_cy),
            })

    return violations


def assign_violations_to_vehicle(all_violations_global, vehicles):
    """
    Gán mỗi violation vào xe máy gần nhất theo tâm đáy.
    Trả về dict: {tuple(bbox): [(vtype, details, bbox, conf), ...]}
    """
    vehicle_violation_map = {tuple(v[0]): [] for v in vehicles}

    for viol in all_violations_global:
        best_vehicle_bbox = None
        best_score = float('inf')

        for v in vehicles:
            v_bbox, v_vtype, _color, _conf, _crop, _crop_bbox = v
            if v_vtype != 'motorcycle':
                continue

            vx1, vy1, vx2, vy2 = v_bbox
            vbx1, vby1, vbx2, vby2 = viol['bbox']
            xi1, yi1 = max(vx1, vbx1), max(vy1, vby1)
            xi2, yi2 = min(vx2, vbx2), min(vy2, vby2)
            inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
            viol_area = max(0, vbx2 - vbx1) * max(0, vby2 - vby1)
            veh_area = max(0, vx2 - vx1) * max(0, vy2 - vy1)
            union_area = veh_area + viol_area - inter_area
            iou = inter_area / union_area if union_area > 0 else 0

            veh_bottom = ((vx1 + vx2) / 2.0, float(vy2))
            viol_bottom = viol['bottom_center']
            dist = np.sqrt((viol_bottom[0] - veh_bottom[0])**2 + (viol_bottom[1] - veh_bottom[1])**2)
            score = dist * 0.5 if iou > 0.1 else dist

            if score < best_score:
                best_score = score
                best_vehicle_bbox = tuple(v_bbox)

        if best_vehicle_bbox is not None:
            vehicle_violation_map[best_vehicle_bbox].append((
                viol['type'], viol['details'], viol['bbox'], viol['conf'],
            ))

    return vehicle_violation_map


def merge_violation_data(vehicle_violation_map, vehicles, matched):
    """Gộp dữ liệu vi phạm vào danh sách xe để hiển thị."""
    all_data = []
    for v in vehicles:
        bbox = v[0]
        violations = vehicle_violation_map.get(tuple(bbox), [])
        if not violations:
            continue
        plate_text = next((m[4] for m in matched if m[0] == bbox), '')
        all_data.append((bbox, v[1], v[2], plate_text, tuple(violations)))
    return all_data


print('✅ Core detection functions ready!')

In [ ]:
# ======================== TRAFFIC LIGHT DETECTION ========================

def detect_traffic_scene(image):
    """
    Phát hiện đèn giao thông và người/phương tiện từ traffic_light.pt.
    Trả về: (lights, road_users)
      - lights: list dict {bbox, class_name, state, conf}
      - road_users: list dict {bbox, class_name, conf}
    """
    lights = []
    road_users = []
    if light_model is None or image is None or image.size == 0:
        return lights, road_users

    results = light_model.predict(image, device='cpu', conf=0.25, verbose=False)
    if len(results) == 0 or results[0].boxes is None:
        return lights, road_users

    names = results[0].names
    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls_id = int(box.cls[0])
        cls_name = names.get(cls_id, str(cls_id))
        conf = float(box.conf[0])
        normalized = cls_name.lower()

        if normalized.startswith('trafficlight'):
            h, w = image.shape[:2]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            light_img = image[y1:y2, x1:x2]
            state = infer_traffic_light_state(cls_name, light_img)
            lights.append({
                'bbox': (x1, y1, x2, y2),
                'class_name': cls_name,
                'state': state,
                'conf': conf
            })
        elif normalized in TRAFFIC_ROAD_USER_CLASSES:
            road_users.append({
                'bbox': (x1, y1, x2, y2),
                'class_name': cls_name,
                'conf': conf
            })
    return lights, road_users


def _classify_traffic_light_by_zones(light_img):
    """Chia ảnh cột đèn thành 3 phần dọc để phân tích màu HSV."""
    if light_img is None or light_img.size == 0:
        return 'unknown'
    h, w = light_img.shape[:2]
    if h < 10:
        return 'unknown'

    top_end = int(h * 0.30)
    mid_end = int(h * 0.65)

    top_zone = light_img[0:top_end, :]
    mid_zone = light_img[top_end:mid_end, :]
    bot_zone = light_img[mid_end:h, :]

    def get_hsv(img):
        return cv2.cvtColor(img, cv2.COLOR_BGR2HSV) if img.size > 0 else None

    hsv_top = get_hsv(top_zone)
    hsv_mid = get_hsv(mid_zone)
    hsv_bot = get_hsv(bot_zone)

    def bright_saturated(hsv_img):
        if hsv_img is None: return None
        return (hsv_img[:, :, 1] >= 60) & (hsv_img[:, :, 2] >= 80)

    def count_color(hsv_img, color_range_fn):
        if hsv_img is None: return 0
        bs = bright_saturated(hsv_img)
        if bs is None: return 0
        mask = color_range_fn(hsv_img[:, :, 0]) & bs
        return int(np.count_nonzero(mask))

    def red_range(h): return (h <= 10) | (h >= 170)
    def yellow_range(h): return (h >= 15) & (h <= 38)
    def green_range(h): return (h >= 40) & (h <= 95)

    top_red = count_color(hsv_top, red_range)
    mid_red = count_color(hsv_mid, red_range)
    bot_red = count_color(hsv_bot, red_range)
    top_yellow = count_color(hsv_top, yellow_range)
    mid_yellow = count_color(hsv_mid, yellow_range)
    bot_yellow = count_color(hsv_bot, yellow_range)
    top_green = count_color(hsv_top, green_range)
    mid_green = count_color(hsv_mid, green_range)
    bot_green = count_color(hsv_bot, green_range)

    top_area = top_zone.shape[0] * top_zone.shape[1]
    mid_area = mid_zone.shape[0] * mid_zone.shape[1]
    bot_area = bot_zone.shape[0] * bot_zone.shape[1]
    min_px_top = max(4, int(top_area * 0.002))
    min_px_mid = max(4, int(mid_area * 0.002))
    min_px_bot = max(4, int(bot_area * 0.002))

    if top_red >= min_px_top and top_red >= max(top_green, top_yellow) * 0.35:
        return 'red'
    if mid_yellow >= min_px_mid and mid_yellow >= max(mid_red, mid_green) * 0.5:
        return 'yellow'
    if bot_green >= min_px_bot and bot_green >= max(bot_red, bot_yellow) * 0.45:
        return 'green'

    total_red = top_red + mid_red + bot_red
    total_green = top_green + mid_green + bot_green
    if top_red >= min_px_top and total_red >= total_green * 0.5:
        return 'red'

    return 'unknown'


def _classify_traffic_light_full(light_img):
    """Phân tích màu trên toàn bộ ảnh (fallback)."""
    if light_img is None or light_img.size == 0:
        return 'unknown'
    hsv = cv2.cvtColor(light_img, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    bright_sat = (s >= 60) & (v >= 80)

    red_mask = (((h <= 10) | (h >= 170)) & bright_sat)
    yellow_mask = ((h >= 15) & (h <= 38) & bright_sat)
    green_mask = ((h >= 40) & (h <= 95) & bright_sat)

    scores = {
        'red': int(np.count_nonzero(red_mask)),
        'yellow': int(np.count_nonzero(yellow_mask)),
        'green': int(np.count_nonzero(green_mask))
    }
    best_state, best_score = max(scores.items(), key=lambda item: item[1])
    signal_pixels = sum(scores.values())

    if signal_pixels == 0:
        return 'unknown'

    crop_area = light_img.shape[0] * light_img.shape[1]
    enough_pixels = best_score >= max(4, int(crop_area * 0.002))
    dominant_enough = best_score / max(1, signal_pixels) >= 0.45
    return best_state if enough_pixels and dominant_enough else 'unknown'


def infer_traffic_light_state(cls_name, light_img):
    """
    Suy luận màu đèn từ class của model, fallback bằng phân tích HSV theo vùng (zone-based).
    """
    normalized = cls_name.lower().replace('_', '-').strip()
    if 'red' in normalized:
        return 'red'

    # Phân tích màu theo vùng (zone-based)
    zone_state = _classify_traffic_light_by_zones(light_img)
    if zone_state != 'unknown':
        return zone_state

    # Fallback: phân tích toàn bộ ảnh
    color_state = _classify_traffic_light_full(light_img)
    if 'yellow' in normalized:
        return 'yellow'
    if 'green' in normalized:
        return 'green'
    return color_state


def red_light_is_active(lights):
    """Kiểm tra xem có đèn đỏ đang bật không."""
    return any(light['state'] == 'red' for light in lights)


print('✅ Traffic light detection functions ready!')

In [ ]:
# ======================== ZONE-BASED RED LIGHT DETECTION ========================

class ZoneConfig:
    """
    Cấu hình các vùng theo tọa độ Y.
    direction: 'down' = top-down (xe đi từ Y nhỏ -> Y lớn)
               'up' = rear-view (xe đi từ Y lớn -> Y nhỏ)
    """
    def __init__(self, direction='down', waiting_end=200, stop_end=300, intersection_end=500):
        self.direction = direction
        self.waiting_start = 0
        self.waiting_end = waiting_end
        self.stop_start = waiting_end
        self.stop_end = stop_end
        self.intersection_start = stop_end
        self.intersection_end = intersection_end

    @property
    def stop_line_y(self):
        return (self.stop_start + self.stop_end) // 2

    def get_zone_name(self, y_bottom):
        if self.direction == 'up':
            if y_bottom >= self.waiting_end: return 'waiting'
            elif y_bottom >= self.stop_start: return 'stop'
            elif y_bottom <= self.intersection_start: return 'intersection'
        else:
            if y_bottom <= self.waiting_end: return 'waiting'
            elif y_bottom <= self.stop_end: return 'stop'
            elif y_bottom >= self.intersection_start: return 'intersection'
        return 'unknown'


class VehicleZoneState:
    """Theo dõi trạng thái vùng của một xe qua các frame."""
    def __init__(self, track_id, initial_zone):
        self.track_id = track_id
        self.zones_visited = [initial_zone]
        self.current_zone = initial_zone
        self.last_seen = time.time()
        self.violation_detected = False
        self.positions_history = []  # (y_bottom, timestamp)

    def update(self, zone, y_bottom):
        self.last_seen = time.time()
        self.positions_history.append((y_bottom, time.time()))
        if zone != self.current_zone:
            if zone not in self.zones_visited:
                self.zones_visited.append(zone)
            self.current_zone = zone

    def has_violated_red_light(self):
        """
        Vi phạm khi: waiting -> stop -> intersection (đi qua cả 3 vùng).
        Hoặc waiting -> intersection trực tiếp (vượt tốc độ cao).
        """
        if self.violation_detected:
            return True

        has_waiting = 'waiting' in self.zones_visited
        has_stop = 'stop' in self.zones_visited
        has_intersection = 'intersection' in self.zones_visited

        if has_waiting and has_stop and has_intersection:
            ordered = [z for z in self.zones_visited if z != 'unknown']
            if len(ordered) >= 3:
                try:
                    iw = ordered.index('waiting')
                    is_ = ordered.index('stop')
                    ii = ordered.index('intersection')
                    if iw < is_ < ii:
                        self.violation_detected = True
                        return True
                except ValueError:
                    pass

        # Trường hợp: waiting -> intersection trực tiếp (vượt tốc độ cao)
        if has_waiting and has_intersection and not has_stop:
            ordered = [z for z in self.zones_visited if z != 'unknown']
            if len(ordered) >= 2:
                try:
                    iw = ordered.index('waiting')
                    ii = ordered.index('intersection')
                    if iw < ii:
                        self.violation_detected = True
                        return True
                except ValueError:
                    pass

        return False

    def is_clearing_intersection(self):
        """Xe đang thoát giao lộ (đã ở intersection từ đầu)."""
        return len(self.zones_visited) == 1 and self.zones_visited[0] == 'intersection'

    def is_active(self, timeout=1.0):
        return time.time() - self.last_seen < timeout


class RedLightZoneDetector:
    """Phát hiện vượt đèn đỏ dựa trên vùng (zone-based)."""
    def __init__(self, zone_config=None):
        self.config = zone_config or ZoneConfig()
        self.tracked_vehicles = {}  # track_id -> VehicleZoneState
        self.red_light_active = False

    def set_red_light(self, is_red):
        self.red_light_active = is_red

    def update_from_traffic_lights(self, lights):
        is_red = any(light.get('state') == 'red' for light in lights)
        self.set_red_light(is_red)

    def process_vehicles(self, road_users):
        """
        Xử lý danh sách xe, trả về danh sách vi phạm.
        road_users: list dict {bbox, class_name, conf}
        """
        violations = []
        current_ids = set()

        for obj in road_users:
            bbox = obj.get('bbox')
            cls_name = obj.get('class_name', 'vehicle')
            conf = obj.get('conf', 0.0)
            if bbox is None or len(bbox) != 4:
                continue

            x1, y1, x2, y2 = bbox
            y_bottom = y2
            zone = self.config.get_zone_name(y_bottom)

            # Tạo track_id từ hash của bbox (cho single image)
            track_id = hash(tuple(bbox)) % 100000
            current_ids.add(track_id)

            if track_id not in self.tracked_vehicles:
                self.tracked_vehicles[track_id] = VehicleZoneState(track_id, zone)

            self.tracked_vehicles[track_id].update(zone, y_bottom)

        # Dọn dẹp các track không còn hoạt động
        inactive = [tid for tid, state in self.tracked_vehicles.items()
                    if not state.is_active()]
        for tid in inactive:
            del self.tracked_vehicles[tid]

        # Kiểm tra vi phạm khi đèn đỏ
        if self.red_light_active:
            for tid, state in self.tracked_vehicles.items():
                if state.violation_detected:
                    continue
                if state.is_clearing_intersection():
                    continue
                if state.has_violated_red_light():
                    # Tìm bbox từ road_users
                    obj_info = None
                    for obj in road_users:
                        obj_tid = hash(tuple(obj.get('bbox', (0,0,0,0)))) % 100000
                        if obj_tid == tid:
                            obj_info = obj
                            break
                    violations.append({
                        'bbox': obj_info.get('bbox') if obj_info else None,
                        'class_name': obj_info.get('class_name', 'vehicle') if obj_info else 'vehicle',
                        'conf': obj_info.get('conf', 0.0) if obj_info else 0.0,
                        'details': 'Vượt đèn đỏ (zone-based)',
                        'track_id': tid,
                        'zone_history': state.zones_visited.copy(),
                        'violation_type': 'RED_LIGHT_VIOLATION',
                    })
                    state.violation_detected = True

        return violations

    def draw_zones(self, frame):
        """Vẽ các vùng lên frame để debug."""
        h, w = frame.shape[:2]
        overlay = frame.copy()

        # Vẽ vùng
        cv2.rectangle(overlay, (0, self.config.waiting_start),
                      (w, self.config.waiting_end), ZONE_COLORS['waiting'], -1)
        cv2.rectangle(overlay, (0, self.config.stop_start),
                      (w, self.config.stop_end), ZONE_COLORS['stop'], -1)
        cv2.rectangle(overlay, (0, self.config.intersection_start),
                      (w, self.config.intersection_end), ZONE_COLORS['intersection'], -1)

        alpha = 0.15
        frame = cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

        # Nhãn
        labels = {
            'WAITING ZONE': (self.config.waiting_start + self.config.waiting_end) // 4,
            'STOP ZONE': (self.config.stop_start + self.config.stop_end) // 2,
            'INTERSECTION': (self.config.intersection_start + self.config.intersection_end) // 2,
        }
        for label, y_center in labels.items():
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
            x_center = (w - tw) // 2
            cv2.putText(frame, label, (x_center, y_center),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

        # Vạch dừng
        stop_y = self.config.stop_line_y
        cv2.line(frame, (0, stop_y), (w, stop_y), (0, 0, 255), 3)
        cv2.putText(frame, 'STOP LINE', (10, stop_y - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

        return frame

    def reset(self):
        self.tracked_vehicles.clear()
        self.red_light_active = False


print('✅ Zone-based red light detection ready!')

In [ ]:
# ======================== TRACKING & DRAWING ========================

def centroid(box):
    return ((box[0] + box[2]) // 2, (box[1] + box[3]) // 2)


def track_vehicles(current, prev, max_dist=100):
    """Centroid tracking: theo dõi xe qua các frame."""
    new, used = [], set()
    active = [t for t in prev if time.time() - t['last_seen'] < 1.0]
    for v in current:
        vc = centroid(v[0])
        best, bd = None, float('inf')
        for i, pt in enumerate(active):
            if i in used:
                continue
            d = np.sqrt((vc[0] - pt['centroid'][0])**2 + (vc[1] - pt['centroid'][1])**2)
            if d < bd and d < max_dist:
                bd, best = d, i
        if best is not None:
            used.add(best)
            t = active[best]
            t['bbox'] = v[0]
            t['centroid'] = vc
            t['last_seen'] = time.time()
            new.append(t)
        else:
            new.append({
                'bbox': v[0],
                'centroid': vc,
                'vtype': v[1],
                'color': v[2],
                'conf': v[3],
                'last_seen': time.time(),
                'crossed': False,
                'track_id': int(time.time() * 1000) % 100000
            })
    return new


def draw_traffic_scene(frame, lights, red_light_violations, stop_line_y=None, show_zones=False, zone_detector=None):
    """Vẽ đèn giao thông và vi phạm lên frame."""
    # Vẽ zones nếu được yêu cầu
    if show_zones and zone_detector is not None:
        frame = zone_detector.draw_zones(frame)

    # Vẽ vạch dừng (nếu không vẽ zones)
    if stop_line_y is not None and not show_zones:
        cv2.line(frame, (0, stop_line_y), (frame.shape[1], stop_line_y), (0, 0, 255), 2)
        cv2.putText(frame, 'STOP LINE', (10, max(25, stop_line_y - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 255), 2)

    # Vẽ đèn giao thông
    for light in lights:
        x1, y1, x2, y2 = light['bbox']
        color = STATE_COLORS.get(light['state'], (180, 180, 180))
        label = f"{light['state'].upper()} {light['conf']*100:.1f}%"
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, label, (x1, max(15, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Vẽ vi phạm vượt đèn đỏ
    for viol in red_light_violations:
        if viol.get('bbox') is None:
            continue
        x1, y1, x2, y2 = viol['bbox']
        color = (0, 0, 255)
        label = f"{viol['class_name']} {viol['conf']*100:.1f}%"
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, label, (x1, max(15, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    return frame


def draw_frame(img, vehicles, matched, all_violation_data, red_light_violations, lights, is_red, zone_detector=None, show_zones=False):
    """Vẽ tất cả thông tin lên frame."""
    frame = img.copy()
    h, w = frame.shape[:2]

    # Vẽ traffic scene (đèn + zones + red light violations)
    frame = draw_traffic_scene(
        frame, lights, red_light_violations,
        stop_line_y=None, show_zones=show_zones, zone_detector=zone_detector
    )

    # Vẽ xe và biển số
    plate_map = {tuple(m[0]): m[4] for m in matched}

    for v in vehicles:
        bbox, vtype, color, conf, _, _ = v
        x1, y1, x2, y2 = bbox
        has_violation = any(item[0] == bbox for item in all_violation_data)
        box_color = (0, 0, 255) if has_violation else (0, 255, 0)
        cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
        cv2.putText(frame, f"{color} {vtype}", (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, box_color, 1)
        plate = plate_map.get(tuple(bbox), '')
        if plate:
            cv2.putText(frame, plate, (x1, y2 + 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Vẽ vi phạm
    for item in all_violation_data:
        for viol in item[4]:
            vtype_v, details, (vx1, vy1, vx2, vy2), vconf = viol
            color_v = VIOLATION_COLORS.get(vtype_v, (0, 165, 255))
            cv2.rectangle(frame, (vx1, vy1), (vx2, vy2), color_v, 2)
            label_v = f"{vtype_v} {vconf*100:.1f}%"
            (tw, th), _ = cv2.getTextSize(label_v, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
            cv2.rectangle(frame, (vx1, vy1 - th - 4), (vx1 + tw + 4, vy1), color_v, -1)
            cv2.putText(frame, label_v, (vx1 + 2, vy1 - 2),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)

    # Status bar
    total_violations = len(all_violation_data) + len(red_light_violations)
    st = f"{'DEN DO' if is_red else 'DEN XANH'} | {len(vehicles)} xe | {len(matched)} bien | {total_violations} VP"
    cv2.putText(frame, st, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                (0, 0, 255) if is_red else (0, 200, 0), 2)

    return frame


print('✅ Drawing functions ready!')

---
## 🖼️ 3. XỬ LÝ ẢNH

Upload ảnh để phát hiện xe, biển số, màu sắc, vi phạm (bao gồm vượt đèn đỏ zone-based).

In [ ]:
def process_image(image_path, show_zones=True):
    """Xử lý một ảnh đơn lẻ."""
    img = cv2.imread(image_path)
    if img is None:
        print(f'❌ Không đọc được: {image_path}')
        return None

    print(f'📷 {os.path.basename(image_path)} ({img.shape[1]}x{img.shape[0]})')
    t0 = time.time()

    # 1. Phát hiện phương tiện
    vehicles = detect_vehicles(img)
    print(f'   🚗 Phương tiện: {len(vehicles)}')

    # 2. Phát hiện biển số
    plates = detect_plates(img)
    matched = match_plates_to_vehicles(vehicles, plates)
    print(f'   🏷️ Biển số: {len(matched)}')

    # 3. Phát hiện vi phạm (full-image)
    all_violations_global = detect_violations_on_full_image(img)
    vehicle_violation_map = assign_violations_to_vehicle(all_violations_global, vehicles)
    all_violation_data = merge_violation_data(vehicle_violation_map, vehicles, matched)
    print(f'   🚨 Vi phạm (helmet): {len(all_violation_data)}')

    # 4. Phát hiện đèn giao thông
    lights, road_users = detect_traffic_scene(img)
    is_red = red_light_is_active(lights)
    print(f'   🔴 Đèn giao thông: {len(lights)} ({ "DO" if is_red else "XANH" })')

    # 5. Phát hiện vượt đèn đỏ (zone-based)
    zone_detector = RedLightZoneDetector(
        ZoneConfig(direction='down', waiting_end=img.shape[0]//3, stop_end=img.shape[0]//2, intersection_end=img.shape[0]*2//3)
    )
    zone_detector.update_from_traffic_lights(lights)
    red_light_violations = zone_detector.process_vehicles(road_users)
    print(f'   🚨 Vượt đèn đỏ: {len(red_light_violations)}')

    # 6. Vẽ kết quả
    result = draw_frame(
        img, vehicles, matched, all_violation_data,
        red_light_violations, lights, is_red,
        zone_detector=zone_detector, show_zones=show_zones
    )

    print(f'⏱️ {time.time()-t0:.2f}s')

    # Hiển thị
    plt.figure(figsize=(14, 10))
    plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    # Lưu
    out_path = f'{DRIVE_BASE}/results/result_{os.path.basename(image_path)}'
    cv2.imwrite(out_path, result)
    print(f'💾 Saved: {out_path}')

    # In chi tiết vi phạm
    if all_violation_data:
        print('\n📋 CHI TIẾT VI PHẠM:')
        for item in all_violation_data:
            bbox, vtype, color, plate_text, violations = item
            print(f'   {color} {vtype} | Biển: {plate_text or "N/A"}')
            for v in violations:
                print(f'      - {v[0]}: {v[1]} (conf: {v[3]*100:.1f}%)')
    if red_light_violations:
        print(f'\n🚨 VƯỢT ĐÈN ĐỎ: {len(red_light_violations)} xe')
        for v in red_light_violations:
            print(f'   - {v["class_name"]} | Zone history: {v.get("zone_history", [])}')

    return result


uploaded = files.upload()
for fname in uploaded:
    process_image(fname, show_zones=True)

---
## 📹 4. XỬ LÝ VIDEO REALTIME (Zone-based Red Light Detection)

Video được stream trực tiếp lên HTML. Sử dụng zone-based detection thay vì stop line đơn giản.

In [ ]:
# === HTML UI: Realtime Video + Log ===
REALTIME_HTML = '''
<div style="display:flex;flex-direction:column;align-items:center;background:#1e1e1e;padding:15px;border-radius:8px;width:870px">
  <div><img id="vf" src="" width="850" style="border-radius:4px"/></div>
  <div style="width:850px;margin-top:15px">
    <h4 style="color:#ff4d4d;margin:5px 0;font-family:sans-serif"> DANH SÁCH PHƯƠNG TIỆN VI PHẠM:</h4>
    <div id="logc" style="background:#000;color:#0f0;font-family:monospace;height:130px;overflow-y:scroll;padding:10px;border:1px solid #333;border-radius:4px;font-size:13px;line-height:1.5">
      [Hệ thống đang khởi động...]<br/>
    </div>
  </div>
</div>
<script>
function al(t){var d=document.getElementById('logc');d.innerHTML+=t+'<br/>';d.scrollTop=d.scrollHeight}
function uf(b){document.getElementById('vf').src='data:image/jpeg;base64,'+b}
</script>
'''

def process_video_realtime(video_path, output_path=None, max_frames=500, show_zones=True):
    """Xử lý video realtime với zone-based red light detection."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f'❌ Không mở được: {video_path}')
        return None

    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Khởi tạo zone detector với config phù hợp
    zone_detector = RedLightZoneDetector(
        ZoneConfig(direction='down', waiting_end=h//3, stop_end=h//2, intersection_end=h*2//3)
    )

    print(f'📹 {os.path.basename(video_path)}: {w}x{h} | Zones: waiting<={h//3}, stop<={h//2}, intersection>={h*2//3}')

    writer = None
    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), min(fps, 15), (w, h))

    clear_output(wait=True)
    display(HTML(REALTIME_HTML), display_id=True)

    prev_tracks = []
    fc, proc = 0, 0
    stats = {
        'vehicles': 0,
        'plates': set(),
        'violations': [],
        'red_light_violations': 0,
        'red_frames': 0
    }
    last_log = {}

    pbar = tqdm(total=min(max_frames, total), desc='Realtime')

    while proc < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        fc += 1
        if fc % 2 != 0:  # Skip every other frame
            continue

        # 1. Phát hiện phương tiện
        vehicles = detect_vehicles(frame, conf=0.3)

        # 2. Phát hiện biển số
        plates = detect_plates(frame)
        matched = match_plates_to_vehicles(vehicles, plates)

        # 3. Phát hiện vi phạm (full-image)
        all_violations_global = detect_violations_on_full_image(frame)
        vehicle_violation_map = assign_violations_to_vehicle(all_violations_global, vehicles)
        all_violation_data = merge_violation_data(vehicle_violation_map, vehicles, matched)

        # 4. Phát hiện đèn giao thông
        lights, road_users = detect_traffic_scene(frame)
        is_red = red_light_is_active(lights)
        zone_detector.update_from_traffic_lights(lights)

        if is_red:
            stats['red_frames'] += 1

        # 5. Phát hiện vượt đèn đỏ (zone-based)
        red_light_violations = zone_detector.process_vehicles(road_users)

        # 6. Tracking
        prev_tracks = track_vehicles(vehicles, prev_tracks)

        # 7. Cập nhật thống kê
        stats['vehicles'] += len(vehicles)
        for m in matched:
            stats['plates'].add(m[4])
        for v in all_violations_global:
            stats['violations'].append(v)
        for v in red_light_violations:
            stats['red_light_violations'] += 1
            now = time.time()
            lk = f"rl_{v.get('track_id', 0)}"
            if lk not in last_log or now - last_log[lk] > 2:
                ts = datetime.now().strftime('%H:%M:%S')
                html = f"<span style='color:#ff4d4d;'>[{ts}] VUOT DEN DO: <b>{v['class_name'].upper()}</b> (conf:{v['conf']:.2f}) | Zones: {v.get('zone_history', [])}</span>"
                display(HTML(f"<script>al('{html}');</script>"), display_id='la')
                last_log[lk] = now

        # 8. Vẽ kết quả
        res = draw_frame(
            frame, vehicles, matched, all_violation_data,
            red_light_violations, lights, is_red,
            zone_detector=zone_detector, show_zones=show_zones
        )

        cv2.putText(res, f'Frame:{proc}', (10, h - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)

        if writer:
            writer.write(res)

        # Stream lên HTML
        _, buf = cv2.imencode('.jpg', res, [cv2.IMWRITE_JPEG_QUALITY, 70])
        display(HTML(f"<script>uf('{base64.b64encode(buf).decode()}');</script>"), display_id='va')

        proc += 1
        pbar.update(1)

    pbar.close()
    cap.release()
    if writer:
        writer.release()

    # In kết quả
    print('\n' + '='*50)
    print('📊 KẾT QUẢ')
    print('='*50)
    print(f'📹 {os.path.basename(video_path)}')
    print(f'📄 Frame xử lý: {proc}/{total}')
    print(f'🔴 Frame đèn đỏ: {stats["red_frames"]}')
    print(f'🚗 Xe: {stats["vehicles"]}')
    print(f'🏷️ Biển số: {", ".join(stats["plates"]) if stats["plates"] else "không"}')
    print(f'🚨 Vượt đèn đỏ (zone-based): {stats["red_light_violations"]}')
    print(f'🚨 Vi phạm khác: {len(stats["violations"])}')
    vc = {}
    for v in stats['violations']:
        vc[v['type']] = vc.get(v['type'], 0) + 1
    for k, v in vc.items():
        print(f'   {k}: {v}')
    if output_path:
        print(f'💾 Video kết quả: {output_path}')

    return stats


def process_video_from_drive(video_name, max_frames=500, show_zones=True):
    """Xử lý video từ Google Drive."""
    p = f'{DRIVE_BASE}/videos/{video_name}'
    if not os.path.exists(p):
        if os.path.exists(video_name):
            p = video_name
        else:
            print(f'❌ Không tìm thấy: {p}')
            return None
    return process_video_realtime(p, f'{DRIVE_BASE}/results/processed_{video_name}', max_frames, show_zones)


def upload_and_process_video(max_frames=300, show_zones=True):
    """Upload và xử lý video."""
    u = files.upload()
    if not u:
        return None
    res = []
    for fn in u:
        print(f'\n📂 {fn}')
        r = process_video_realtime(fn, f'{DRIVE_BASE}/results/processed_{fn}', max_frames, show_zones)
        if r:
            res.append(r)
    return res


print('✅ Video REALTIME ready!')
print()
print('📌 Chạy: upload_and_process_video()')

In [ ]:
# === CHẠY TEST VIDEO REALTIME ===
result = upload_and_process_video(max_frames=500, show_zones=True)

---
## 🌍 5. BIRD'S EYE VIEW (BEV) 3D - PHÁT HIỆN VƯỢT ĐÈN ĐỎ KHÔNG GIAN 3D

Sử dụng **Perspective Transform** để chuyển đổi không gian ảnh 2D sang không gian 3D (Bird's Eye View).

### 🎯 Cách hoạt động:
1. **Vùng hình thang (Trapezoid)**: 4 điểm src_points đánh dấu vùng làn đường cần giám sát
2. **Perspective Transform**: Chiếu vùng hình thang thành hình chữ nhật trong BEV space
3. **Vạch dừng ảo (Stop Line 3D)**: Đường thẳng Y=400 trong BEV space, chiếu ngược về ảnh gốc
4. **Buffer đèn đỏ 30 frame**: Chống nhấp nháy khi phát hiện đèn
5. **Theo dõi vi phạm**: Xe nào vượt vạch khi đèn đỏ → officially_violated_ids (giữ khung đỏ mãi mãi)
6. **Log 1 lần duy nhất**: logged_violated_ids chống trùng lặp thông báo

### 🖊️ CẤU HÌNH:
Bạn **CẦN** điều chỉnh `src_points` sao cho khớp với làn đường trong video của bạn!
- 4 điểm: [bottom-left, bottom-right, top-right, top-left]
- Dùng công cụ paint/photo để lấy tọa độ pixel chính xác từ frame đầu tiên của video

In [ ]:
# ======================== BIRD'S EYE VIEW (BEV) 3D DETECTOR ========================
# Code chuyển từ backend/src/core/birds_eye_detector.py, giữ nguyên logic Colab gốc

class BirdsEyeRedLightDetector:
    """
    Phát hiện vượt đèn đỏ sử dụng Perspective Transform sang BEV.
    
    Tính năng:
      - Chuyển đổi tọa độ 2D -> 3D BEV
      - Phát hiện vượt vạch dừng ảo trong BEV space
      - Chỉ báo vi phạm 1 lần duy nhất cho mỗi xe
      - Buffer đèn đỏ chống nhấp nháy (30 frames)
      - Ghim khung đỏ vĩnh viễn cho xe vi phạm
      - Trạng thái 'DỪNG CHỜ' khi đèn đỏ chưa vượt vạch
    """

    def __init__(self, src_points=None, dst_points=None, stop_line_3d_y=400, red_light_buffer_frames=30):
        # 4 điểm nguồn trong không gian ảnh gốc (trapezoid)
        self.src_points = np.float32([
            [518, 705],    # 1. Duoi - Trai
            [1444, 804],   # 2. Duoi - Phai
            [1480, 602],   # 3. Tren - Phai
            [835, 600]     # 4. Tren - Trai
        ]) if src_points is None else np.float32(src_points)

        # 4 điểm đích trong không gian 3D BEV (rectangle)
        self.dst_points = np.float32([
            [0, 600],     # Duoi - Trai
            [400, 600],   # Duoi - Phai
            [400, 0],     # Tren - Phai
            [0, 0]        # Tren - Trai
        ]) if dst_points is None else np.float32(dst_points)

        # Perspective transform matrices
        self.M = cv2.getPerspectiveTransform(self.src_points, self.dst_points)
        self.M_inv = np.linalg.inv(self.M)

        # Vạch dừng trong BEV space
        self.STOP_LINE_3D_Y = stop_line_3d_y

        # Red light state with buffer (giống Colab)
        self.red_light_counter = 0
        self.RED_LIGHT_BUFFER_FRAMES = red_light_buffer_frames

        # Violation tracking (giống Colab)
        self.officially_violated_ids = set()   # Xe DA BI Phat - giu khung do mai mai
        self.logged_violated_ids = set()       # Xe DA IN LOG - chong trung lap

        # Stats
        self.frame_count = 0

    def convert_to_3d_point(self, cx, cy):
        """Chuyển đổi tọa độ 2D (image space) sang 3D (BEV space)."""
        point = np.array([[[cx, cy]]], dtype=np.float32)
        transformed = cv2.perspectiveTransform(point, self.M)
        return transformed[0][0]

    def get_stop_line_2d(self):
        """Tính vạch dừng trong image space từ BEV stop line."""
        line_3d_pts = np.array([
            [[0, self.STOP_LINE_3D_Y]],
            [[400, self.STOP_LINE_3D_Y]]
        ], dtype=np.float32)
        line_2d_pts = cv2.perspectiveTransform(line_3d_pts, self.M_inv)
        left = (int(line_2d_pts[0][0][0]), int(line_2d_pts[0][0][1]))
        right = (int(line_2d_pts[1][0][0]), int(line_2d_pts[1][0][1]))
        return left, right

    def is_point_in_bev_zone(self, x_3d, y_3d):
        """Kiểm tra điểm có nằm trong vùng BEV hợp lệ không."""
        return (0 <= x_3d <= 400) and (0 <= y_3d <= 600)

    def update_red_light(self, lights):
        """
        Cập nhật trạng thái đèn đỏ với buffer chống nhấp nháy.
        lights: list từ detect_traffic_scene() hoặc model.track()
        """
        # Phát hiện đèn đỏ từ class name
        yolo_detected_red = False
        for light in lights:
            cls_name = light.get('class_name', '').lower()
            if 'trafficlight' in cls_name and 'red' in cls_name:
                yolo_detected_red = True
                break

        if yolo_detected_red:
            self.red_light_counter = self.RED_LIGHT_BUFFER_FRAMES
        else:
            if self.red_light_counter > 0:
                self.red_light_counter -= 1

    @property
    def is_red_light_active(self):
        return self.red_light_counter > 0

    def process_vehicle(self, track_id, class_name, bbox):
        """
        Xử lý một xe và kiểm tra vi phạm đèn đỏ.
        Giống Colab:
          1. Tính center_x và bottom_y (cx, cy_tail)
          2. Chuyển đổi sang BEV (x_3d, y_3d)
          3. Kiểm tra trong zone và qua vạch
          4. Nếu đèn đỏ + qua vạch -> VI PHẠM
          5. officially_violated_ids: thêm vào danh sách đen
          6. logged_violated_ids: log 1 lần duy nhất
        """
        x1, y1, x2, y2 = bbox
        cx = int((x1 + x2) / 2)
        cy_tail = y2

        # Chuyển đổi sang BEV
        x_3d, y_3d = self.convert_to_3d_point(cx, cy_tail)
        unique_key = f"{class_name}_{track_id}"

        # Kiểm tra trong zone
        is_inside_zone = self.is_point_in_bev_zone(x_3d, y_3d)
        is_past_line_3d = is_inside_zone and (y_3d < self.STOP_LINE_3D_Y)

        color = (0, 255, 0)
        label = unique_key
        is_violation = False
        is_first_violation = False
        show_waiting_label = False

        # Nếu xe đã từng bị bắt vi phạm trước đó
        if unique_key in self.officially_violated_ids:
            is_violation = True

        # Nếu đèn đỏ và xe mới vượt vạch lần đầu
        elif self.is_red_light_active and is_past_line_3d:
            self.officially_violated_ids.add(unique_key)
            is_violation = True

            if unique_key not in self.logged_violated_ids:
                self.logged_violated_ids.add(unique_key)
                is_first_violation = True

        # Đèn đỏ nhưng xe chưa vượt vạch
        elif self.is_red_light_active:
            show_waiting_label = True

        if is_violation:
            color = (0, 0, 255)
            label = f"VI PHAM 3D! {unique_key.upper()}"
        elif show_waiting_label:
            color = (0, 255, 255)
            label = f"DUNG CHO: {unique_key}"

        return {
            'unique_key': unique_key,
            'is_inside_zone': is_inside_zone,
            'is_past_line_3d': is_past_line_3d,
            'bev_coords': (float(x_3d), float(y_3d)),
            'is_violation': is_violation,
            'first_time_violation': is_first_violation,
            'show_waiting_label': show_waiting_label,
            'color': color,
            'label': label,
        }

    def draw_all(self, frame, tracked_vehicles, vehicle_classes):
        """
        Vẽ tất cả overlay BEV lên frame (giống Colab):
          1. Vùng hình thang kiểm soát (màu xanh dương)
          2. Vạch dừng (màu cam)
          3. Label cho từng xe (VI PHẠM / DỪNG CHỜ / xanh)
          4. Trạng thái hệ thống
        """
        # 1. Vẽ vùng hình thang kiểm soát
        pts = self.src_points.astype(np.int32)
        cv2.polylines(frame, [pts], True, (255, 255, 0), 2)

        # 2. Vẽ vạch dừng ảo
        try:
            left, right = self.get_stop_line_2d()
            cv2.line(frame, left, right, (0, 152, 255), 4)
        except Exception:
            pass

        # 3. Xử lý từng xe
        for trk_id, bbox, cls_id, conf in tracked_vehicles:
            x1, y1, x2, y2 = bbox
            class_name = vehicle_classes.get(cls_id, 'vehicle')

            result = self.process_vehicle(trk_id, class_name, bbox)
            color = result['color']
            label = result['label']
            unique_key = result['unique_key']

            # Nếu vi phạm, vẽ thêm label nền đỏ
            if result['is_violation']:
                cv2.rectangle(frame, (x1, y1 - 20), (x1 + 180, y1), (0, 0, 255), -1)
                cv2.putText(frame, label, (x1 + 5, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
            else:
                cv2.putText(frame, label, (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        # 4. Hiển thị trạng thái hệ thống
        status_text = 'HE THONG 3D: DANG DEN DO' if self.is_red_light_active else 'HE THONG 3D: DEN XANH / VANG'
        status_color = (0, 0, 255) if self.is_red_light_active else (0, 255, 0)
        cv2.putText(frame, status_text, (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)

        return frame

    def reset(self):
        self.red_light_counter = 0
        self.officially_violated_ids.clear()
        self.logged_violated_ids.clear()
        self.frame_count = 0

    def get_stats(self):
        return {
            'frame_count': self.frame_count,
            'red_light_active': self.is_red_light_active,
            'red_light_counter': self.red_light_counter,
            'total_violations': len(self.officially_violated_ids),
            'total_logged': len(self.logged_violated_ids),
        }


print('✅ BEV 3D Detector ready!')
print()
print('📌 CÁCH DÙNG:')
print('   # Bước 1: Khởi tạo detector với src_points phù hợp')
print('   bev = BirdsEyeRedLightDetector(src_points=your_4_points)')
print()
print('   # Bước 2: Trong vòng lặp xử lý frame:')
print('   lights, _ = detect_traffic_scene(frame)')
print('   bev.update_red_light(lights)')
print('   frame = bev.draw_all(frame, tracked, VEHICLE_CLASSES)')
print('')

---
## 🎬 6. CHẠY BEV 3D TRÊN VIDEO

Cell này sử dụng `traffic_light.pt` với `model.track()` và vùng hình thang BEV 3D để phát hiện vượt đèn đỏ.

In [ ]:
# ======================== BEV 3D HTML UI ========================
BEV_HTML = '''
<div style="display:flex;flex-direction:column;align-items:center;background:#1e1e1e;padding:15px;border-radius:8px;width:870px">
  <div><img id="bev_frame" src="" width="850" style="border-radius:4px"/></div>
  <div style="width:850px;margin-top:15px">
    <h4 style="color:#00aaff;margin:5px 0;font-family:sans-serif"> HỆ THỐNG GIÁM SÁT KHÔNG GIAN 3D (BIRD'S EYE VIEW):</h4>
    <div id="bev_log" style="background:#000;color:#00ff00;font-family:monospace;height:160px;overflow-y:scroll;padding:10px;border:1px solid #333;border-radius:4px;text-align:left;font-size:13px;line-height:1.6">
      [Đã thiết lập hệ thống ghi nhận vi phạm độc lập - Cảnh báo 1 lần duy nhất]...<br/>
    </div>
  </div>
</div>
<script>
function bevLog(text){var d=document.getElementById('bev_log');d.innerHTML+=text+'<br/>';d.scrollTop=d.scrollHeight}
function bevFrame(b){document.getElementById('bev_frame').src='data:image/jpeg;base64,'+b}
</script>
'''

print('✅ BEV HTML UI ready!')
print()
print('📌 Chạy cell bên dưới để xử lý video với BEV 3D')

In [ ]:
# === BEV 3D: XỬ LÝ VIDEO REALTIME (giống code Colab) ===

# 🖊️ ADJUST THESE POINTS TO MATCH YOUR VIDEO:
# Dùng paint / photoshop để lấy tọa độ 4 góc làn đường từ frame đầu video
MY_SRC_POINTS = [
    [518, 705],    # 1. Duoi - Trai (bottom-left)
    [1444, 804],   # 2. Duoi - Phai (bottom-right)
    [1480, 602],   # 3. Tren - Phai (top-right)
    [835, 600]     # 4. Tren - Trai (top-left)
]

def run_bev_on_video(video_path, output_path=None, max_frames=500):
    """
    Xử lý video với BEV 3D detection.
    Sử dụng model.track() cho tracking + perspective transform.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f'❌ Không mở được: {video_path}')
        return None

    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f'📹 {os.path.basename(video_path)}: {w}x{h}')
    print(f'📍 src_points: {MY_SRC_POINTS}')

    # Khởi tạo BEV detector với src_points
    bev = BirdsEyeRedLightDetector(src_points=MY_SRC_POINTS)

    writer = None
    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), min(fps, 15), (w, h))

    clear_output(wait=True)
    display(HTML(BEV_HTML), display_id=True)

    fc, proc = 0, 0
    pbar = tqdm(total=min(max_frames, total), desc='BEV 3D')

    while proc < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        fc += 1
        if fc % 2 != 0:  # Skip every other frame
            continue

        # Dùng model.track() cho detection + tracking (giống Colab)
        results = light_model.track(source=frame, conf=0.20, persist=True, verbose=False)

        # Thu thập thông tin đèn giao thông
        lights = []
        tracked = []

        for r in results:
            if r.boxes is None:
                continue
            for box in r.boxes:
                cls = int(box.cls[0])
                name = light_model.names[cls]
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                track_id = int(box.id[0]) if box.id is not None else 0

                if name.lower().startswith('trafficlight'):
                    # Phân tích trạng thái đèn
                    light_img = frame[y1:y2, x1:x2]
                    state = infer_traffic_light_state(name, light_img)
                    lights.append({
                        'bbox': (x1, y1, x2, y2),
                        'class_name': name,
                        'state': state,
                        'conf': conf
                    })
                else:
                    # Xe / phương tiện
                    cls_id = None
                    for k, v in VEHICLE_CLASSES.items():
                        if v in name.lower():
                            cls_id = k
                            break
                    if cls_id is None and name.lower() in TRAFFIC_ROAD_USER_CLASSES:
                        cls_id = 2  # car as default
                    if cls_id is not None:
                        tracked.append((track_id, (x1, y1, x2, y2), cls_id, conf))

        # Cập nhật trạng thái đèn đỏ với buffer (giống Colab)
        bev.update_red_light(lights)

        # Vẽ BEV lên frame (hình thang + vạch dừng + label) (giống Colab)
        frame = bev.draw_all(frame, tracked, VEHICLE_CLASSES)

        # Vẽ thêm thông tin đèn giao thông
        for light in lights:
            x1, y1, x2, y2 = light['bbox']
            col = STATE_COLORS.get(light['state'], (180, 180, 180))
            cv2.rectangle(frame, (x1, y1), (x2, y2), col, 2)
            cv2.putText(frame, f"{light['state'].upper()} {light['conf']*100:.1f}%",
                        (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2)

        cv2.putText(frame, f'BEV Frame:{proc}', (10, h - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)

        if writer:
            writer.write(frame)

        # Stream lên HTML
        _, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 70])
        display(HTML(f"<script>bevFrame('{base64.b64encode(buf).decode()}');</script>"), display_id='bev_va')

        # Log vi phạm mới - chỉ 1 lần duy nhất (giống Colab)
        for trk_id, bbox, cls_id, conf in tracked:
            class_name = VEHICLE_CLASSES.get(cls_id, 'vehicle')
            result = bev.process_vehicle(trk_id, class_name, bbox)
            if result['first_time_violation']:
                ts = datetime.now().strftime('%H:%M:%S')
                log_text = f"<span style='color:#ff4d4d;'>[{ts}] BEV 3D: Xe <b>{result['unique_key'].upper()}</b> vuot ranh gioi 3D khi den do! (y_3d={result['bev_coords'][1]:.0f})</span>"
                display(HTML(f"<script>bevLog('{log_text}');</script>"), display_id='bev_la')

        proc += 1
        pbar.update(1)

    pbar.close()
    cap.release()
    if writer:
        writer.release()

    # In kết quả
    stats = bev.get_stats()
    print('\n' + '='*50)
    print('📊 KẾT QUẢ BEV 3D')
    print('='*50)
    print(f'📹 {os.path.basename(video_path)}')
    print(f'📄 Frame xử lý: {proc}/{total}')
    print(f'🔴 Đèn đỏ active: {stats["red_light_active"]}')
    print(f'🚨 Tổng vi phạm BEV: {stats["total_violations"]}')
    print(f'📝 Đã log: {stats["total_logged"]}')
    if output_path:
        print(f'💾 Video kết quả: {output_path}')

    return stats


def run_bev_on_drive_video(video_name, max_frames=500):
    """Xử lý video BEV từ Google Drive."""
    p = f'{DRIVE_BASE}/videos/{video_name}'
    if not os.path.exists(p):
        if os.path.exists(video_name):
            p = video_name
        else:
            print(f'❌ Không tìm thấy: {p}')
            return None
    return run_bev_on_video(p, f'{DRIVE_BASE}/results/bev_{video_name}', max_frames)


def upload_and_run_bev(max_frames=300):
    """Upload và xử lý video BEV."""
    u = files.upload()
    if not u:
        return None
    res = []
    for fn in u:
        print(f'\n📂 {fn}')
        r = run_bev_on_video(fn, f'{DRIVE_BASE}/results/bev_{fn}', max_frames)
        if r:
            res.append(r)
    return res


print('✅ BEV 3D Video processing ready!')
print()
print('📌 CHẠY: upload_and_run_bev()')

In [ ]:
# === CHẠY BEV 3D TRÊN VIDEO ===
# 🖊️ Nhớ chỉnh MY_SRC_POINTS ở cell trên trước khi chạy!
result = upload_and_run_bev(max_frames=500)